# RAG Benchmarking: Evaluating External Knowledge Bases

This test evaluates how different HuggingFace datasets perform as a knowledge base. Using game questions from our log data as user queries, we look at what context each dataset would retrieve and inject into an AI prompt.

Note: This is a retrieval-only test. **No LLM calls are made**.

## 1. Imports

In [1]:
import json
import hashlib
import pickle
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
from rank_bm25 import BM25Okapi
from datasets import load_dataset

RUNS_DIR = Path('./data/runs')
CACHE_DIR = Path('./data/index_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Use sklearn's built-in English stopwords — no hand-writing, no downloads
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as STOPWORDS

def clean_tokens(text):
    tokens = text.lower().split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

/home/lillaro/Documents/Programmazione/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Game Questions (Queries) with Answer Info

In [2]:
run_folders = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir()])
latest_run = run_folders[-1]
comp_files = sorted(latest_run.glob('comp_*.json'))

all_q_records = []
for cf in comp_files:
    with open(cf) as f:
        comp = json.load(f)
    for attempt in comp['attempts']:
        for q in attempt['questions']:
            q['comp_name'] = comp['competition_name']
            all_q_records.append(q)

# Find correct answer for each question by looking across all attempts
# If ANY attempt got it right -> chosen_letter is the ground truth
known_answers = {}  # question_text -> correct_letter
for q in all_q_records:
    if q.get('correct') is True:
        known_answers.setdefault(q['question_text'].strip(), q['chosen_letter'])
    else:
        known_answers.setdefault(q['question_text'].strip(), None)

# Deduplicate but keep the first occurrence that has a known answer if possible
seen = set()
unique_questions = []
for q in all_q_records:
    key = q['question_text'].strip()
    if key not in seen:
        seen.add(key)
        q['known_correct_letter'] = known_answers.get(key)
        unique_questions.append(q)

print(f'Loaded {len(unique_questions)} unique questions')
n_known = sum(1 for q in unique_questions if q['known_correct_letter'] is not None)
print(f'  Correct answer known: {n_known}')
print(f'  Correct answer unknown: {len(unique_questions) - n_known}')

# Pick one test query per competition (prefer one with known answer)
queries = {}
for q in unique_questions:
    comp = q['comp_name']
    if comp not in queries:
        queries[comp] = q
    # replace with a known-answer version if found later
    if queries[comp]['known_correct_letter'] is None and q['known_correct_letter'] is not None:
        queries[comp] = q

Loaded 158 unique questions
  Correct answer known: 119
  Correct answer unknown: 39


## 3. Show the Test Queries

In [3]:
for comp, q in queries.items():
    opts = q['options']
    print(f'\nQuery [{comp}]: {q["question_text"]}')
    for letter in ['A', 'B', 'C', 'D']:
        if letter in opts:
            marker = ' ★ CORRECT' if q['known_correct_letter'] == letter else ''
            print(f'    {letter}: {opts[letter]}{marker}')
    if q['known_correct_letter'] is None:
        print(f'    (correct answer unknown from log data)')


Query [Entertainment]: What is the primary goal of the LazyTown series?
    A: To encourage healthy lifestyles ★ CORRECT
    B: To tell stories about space exploration
    C: To promote a new brand of sports equipment
    D: To teach children about science

Query [Ancient History and Politics]: What was the primary reason the construction of the Temple of Olympian Zeus was left unfinished for over 330 years?
    A: Political instability and changes in leadership ★ CORRECT
    B: The temple was damaged by a natural disaster
    C: A lack of funding
    D: A decision to use the site for another purpose

Query [Science and Nature]: What is an energy change that takes place in a light bulb?
    A: Electrical energy changes to light energy. ★ CORRECT
    B: Chemical energy changes to light energy.
    C: Electrical energy changes to chemical energy.
    D: Chemical energy changes to heat energy.

Query [Maths]: What is the ones digit of $1 \cdot 2 \cdot 3 \cdot 4 \cdot 5 \cdot 6 \cdot 7 \c

## 4. Define Datasets to Test

In [4]:
DATASETS = {
    # 'squad': {
    #     'path': 'squad',
    #     'split': 'train',
    #     'text_field': 'context',
    #     'desc': 'SQuAD: Wikipedia passages (full corpus)',
    #     'dedupe': True,
    # },
    # 'rag-datasets/rag-mini-wikipedia': {
    #     'path': 'rag-datasets/rag-mini-wikipedia',
    #     'subset': 'text-corpus',
    #     'split': 'passages',
    #     'text_field': 'passage',
    #     'desc': 'WikiQA: Wikipedia-derived short answers (full corpus)',
    # },
    'aadityaubhat/GPT-wiki-intro': {
        'path': 'aadityaubhat/GPT-wiki-intro',
        'split': 'train',
        'text_field': 'wiki_intro',
        'desc': 'Dataset used for GPT training for text generation',
    },
    # 'cais/mmlu (quiz-questions)': {
    #     'path': 'cais/mmlu',
    #     'split': 'auxiliary_train',
    #     'text_field': 'question',
    #     'desc': 'MMLU: 57-subject multiple-choice questions — closest to our quiz format',
    #     'subset': 'all',
    #     'dedupe' : True
    # },
}

## 5. Build Corpus from a Dataset

In [5]:
def load_corpus(dataset_name, cfg):
    print(f'\nLoading {dataset_name}...')
    split = cfg['split']
    subset = cfg.get('subset')
    dedupe = cfg.get('dedupe', False)

    ds = load_dataset(cfg['path'], subset, split=split)

    texts = ds[cfg['text_field']]

    if dedupe:
        seen = set()
        texts = [t for t in texts if not (t in seen or seen.add(t))]

    texts = [t for t in texts if t and len(t) > 10]
    print(f'  {len(texts)} documents loaded')
    return texts


def _cache_key(dataset_name, cfg):
    raw = f'{dataset_name}|{cfg["path"]}|{cfg["split"]}|{cfg.get("subset","")}|{cfg["text_field"]}|{cfg.get("dedupe",False)}'
    return hashlib.sha256(raw.encode()).hexdigest()[:16]


def build_indexes(corpus_docs):
    print('  Encoding with sentence-transformer...')
    bi_enc = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
    doc_embeddings = bi_enc.encode(
        corpus_docs, convert_to_numpy=True,
        show_progress_bar=False, normalize_embeddings=True,
    )

    dim = doc_embeddings.shape[1]
    index_hnsw = hnswlib.Index(space='cosine', dim=dim)
    index_hnsw.init_index(max_elements=len(corpus_docs), ef_construction=200, M=32)
    index_hnsw.add_items(doc_embeddings, list(range(len(corpus_docs))))
    print(f'  Dense index built (dim={dim})')

    print('  Building BM25...')
    tokenized = [clean_tokens(d) for d in corpus_docs]
    bm25 = BM25Okapi(tokenized)
    print('  BM25 index built')

    return bi_enc, index_hnsw, bm25, doc_embeddings


def load_or_build_indexes(dataset_name, cfg, corpus_docs):
    key = _cache_key(dataset_name, cfg)
    emb_path = CACHE_DIR / f'{key}_emb.npy'
    hnsw_path = CACHE_DIR / f'{key}_hnsw.bin'
    bm25_path = CACHE_DIR / f'{key}_bm25.pkl'

    if emb_path.exists() and hnsw_path.exists() and bm25_path.exists():
        print('  Loading cached indexes...')
        bi_enc = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
        doc_embeddings = np.load(emb_path)
        index_hnsw = hnswlib.Index(space='cosine', dim=doc_embeddings.shape[1])
        index_hnsw.load_index(str(hnsw_path))
        with open(bm25_path, 'rb') as f:
            bm25 = pickle.load(f)
        print(f'  Loaded dense ({doc_embeddings.shape[1]}d, {len(corpus_docs)} docs) + BM25')
        return bi_enc, index_hnsw, bm25

    bi_enc, index_hnsw, bm25, doc_embeddings = build_indexes(corpus_docs)

    print('  Saving indexes to cache...')
    np.save(emb_path, doc_embeddings)
    index_hnsw.save_index(str(hnsw_path))
    with open(bm25_path, 'wb') as f:
        pickle.dump(bm25, f)
    print(f'  Cached to {key}_*.npy/.bin/.pkl')
    return bi_enc, index_hnsw, bm25


def search_dense(query, bi_enc, index_hnsw, corpus_docs, top_k=3):
    q_emb = bi_enc.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    # hnswlib returns cosine DISTANCE (0=identical, 2=opposite)
    # convert to similarity: sim = 1 - dist
    inds, distances = index_hnsw.knn_query(q_emb, k=min(top_k, len(corpus_docs)))
    return [
        {'doc': corpus_docs[int(i)], 'score': 1.0 - float(d)}
        for i, d in zip(inds[0], distances[0])
    ]


def search_sparse(query, bm25, corpus_docs, top_k=3):
    scores = bm25.get_scores(clean_tokens(query))
    top = np.argsort(scores)[::-1][:min(top_k, len(corpus_docs))]
    return [
        {'doc': corpus_docs[int(i)], 'score': float(scores[i])}
        for i in top
    ]


def search_hybrid(query, bi_enc, index_hnsw, bm25, corpus_docs, top_k=3):
    """Reciprocal Rank Fusion of dense + sparse."""
    dense = search_dense(query, bi_enc, index_hnsw, corpus_docs, top_k=top_k * 3)
    sparse = search_sparse(query, bm25, corpus_docs, top_k=top_k * 3)

    rrf_scores = {}
    for rank, r in enumerate(dense):
        rrf_scores[r['doc']] = rrf_scores.get(r['doc'], 0) + 1.0 / (60 + rank + 1)
    for rank, r in enumerate(sparse):
        rrf_scores[r['doc']] = rrf_scores.get(r['doc'], 0) + 1.0 / (60 + rank + 1)

    sorted_docs = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [
        {'doc': d, 'score': rrf_scores[d]}
        for d in sorted_docs
    ]

## 6. Run Retrieval — Every Dataset × Every Query

In [6]:
for ds_name, cfg in DATASETS.items():
    print(f'\n{"#"*70}')
    print(f'## DATASET: {ds_name}')
    print(f'## {cfg["desc"]}')
    print(f'{"#"*70}')

    corpus_docs = load_corpus(ds_name, cfg)
    if len(corpus_docs) < 5:
        print('  SKIP — too few docs')
        continue

    bi_enc, index_hnsw, bm25 = load_or_build_indexes(ds_name, cfg, corpus_docs)

    for comp_name, q_obj in queries.items():
        query = q_obj['question_text']
        opts = q_obj['options']
        print(f'\n  ─── Query [{comp_name}] ───')
        print(f'  Q: {query}')
        for letter in ['A', 'B', 'C', 'D']:
            if letter in opts:
                marker = ' [✓]' if q_obj['known_correct_letter'] == letter else ''
                print(f'     {letter}: {opts[letter]}{marker}')

        for method, search_fn in [
            ('DENSE', lambda q: search_dense(q, bi_enc, index_hnsw, corpus_docs)),
            ('BM25',  lambda q: search_sparse(q, bm25, corpus_docs)),
            ('HYBRID', lambda q: search_hybrid(q, bi_enc, index_hnsw, bm25, corpus_docs)),
        ]:
            print(f'\n{"="*80}')
            print(f'  METHOD: {method}')
            print(f'{"="*80}')

            all_results = []  # accumulate across all sub-queries

            sub_queries = [('QUESTION_ONLY', query)]

            for step_label, step_query in sub_queries:
                print(f'\n{"─"*70}')
                print(f'  SUB-QUERY: {step_label}')
                print(f'  Query text: {step_query}')
                print(f'{"─"*70}')

                results = search_fn(step_query)
                print(f'  Retrieved {len(results)} documents:')
                for i, r in enumerate(results, 1):
                    preview = r['doc'][:200].replace('\n', ' | ')
                    print(f'    #{i} (score={r["score"]:.4f}) {preview}...')

                all_results.extend(results)

            # --- Deduplicate by document text (keep first occurrence) ---
            seen_docs = set()
            deduped = []
            for r in all_results:
                if r['doc'] not in seen_docs:
                    seen_docs.add(r['doc'])
                    deduped.append(r)
            n_dupes = len(all_results) - len(deduped)
            if n_dupes:
                print(f'\n  >>> Removed {n_dupes} duplicate documents')

            # --- Build final combined context ---
            docs_context = '\n\n---\n\n'.join(r['doc'] for r in deduped)
            print(f'\n{"*"*70}')
            print(f'  >>> FINAL COMBINED CONTEXT for [{method}] ({len(deduped)} unique docs, ~{len(docs_context)} chars)')
            print(f'{"*"*70}')
            print(f'  >>> Full context:')
            print(docs_context)
            print(f'  >>> END CONTEXT [{method}]')
            print(f'{"*"*70}')


######################################################################
## DATASET: aadityaubhat/GPT-wiki-intro
## Dataset used for GPT training for text generation
######################################################################

Loading aadityaubhat/GPT-wiki-intro...


Generating train split: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 150000/150000 [00:06<00:00, 22932.21 examples/s]


  150000 documents loaded
  Encoding with sentence-transformer...


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 9388.94it/s]


  Dense index built (dim=384)
  Building BM25...
  BM25 index built
  Saving indexes to cache...
  Cached to 0b9b85eef906381a_*.npy/.bin/.pkl

  ─── Query [Entertainment] ───
  Q: What is the primary goal of the LazyTown series?
     A: To encourage healthy lifestyles [✓]
     B: To tell stories about space exploration
     C: To promote a new brand of sports equipment
     D: To teach children about science

  METHOD: DENSE

──────────────────────────────────────────────────────────────────────
  SUB-QUERY: QUESTION_ONLY
  Query text: What is the primary goal of the LazyTown series?
──────────────────────────────────────────────────────────────────────
  Retrieved 3 documents:
    #1 (score=0.3752) Fastpoint Games was a developer of data-driven games for businesses in the fortune 500, and was the parent company of fantasy sports developer, RotoHog. Under the Fastpoint Games banner, the company h...
    #2 (score=0.3315) Young Social Innovators (YSI) is a "social awareness and active c

## 7. Summary

In [7]:
print('\n\n')
print('=' * 70)
print('SUMMARY')
print('=' * 70)
for comp_name, q_obj in queries.items():
    opts = q_obj['options']
    correct = q_obj['known_correct_letter']
    if correct:
        print(f'\n{comp_name}: "{q_obj["question_text"][:60]}..."')
        print(f'  Correct answer: {correct} → {opts[correct]}')
    else:
        print(f'\n{comp_name}: "{q_obj["question_text"][:60]}..."')
        print(f'  Correct answer: unknown')
        for letter in ['A', 'B', 'C', 'D']:
            print(f'    {letter}: {opts[letter]}')
    # note which dataset had best match
    print(f'  (See above for per-dataset retrieved context)')

print('\nNo LLM calls — retrieval only.')




SUMMARY

Entertainment: "What is the primary goal of the LazyTown series?..."
  Correct answer: A → To encourage healthy lifestyles
  (See above for per-dataset retrieved context)

Ancient History and Politics: "What was the primary reason the construction of the Temple o..."
  Correct answer: A → Political instability and changes in leadership
  (See above for per-dataset retrieved context)

Science and Nature: "What is an energy change that takes place in a light bulb?..."
  Correct answer: A → Electrical energy changes to light energy.
  (See above for per-dataset retrieved context)

Maths: "What is the ones digit of $1 \cdot 2 \cdot 3 \cdot 4 \cdot 5..."
  Correct answer: A → 0
  (See above for per-dataset retrieved context)

No LLM calls — retrieval only.
